In [1]:
! git clone https://github.com/qiuqiangkong/audioset_tagging_cnn.git 
! cd audioset_tagging_cnn/pytorch

Cloning into 'audioset_tagging_cnn'...


In [1]:
import sys
sys.path.append("audioset_tagging_cnn/pytorch")

#from models import Cnn6
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
# Torch hub is one thing
from torch.hub import load
import math
import torch.optim as optim


In [2]:
import os

In [4]:
class Cnn6FrameEmb(Cnn6):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def forward(self, x):
        # Run through convolutional feature extractor
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.conv_block4(x)

        
        # Instead of global pooling → return frame-level embeddings
        # Shape: (batch, channels, time, freq)
        return x

# Build model and load weights
frame_model = Cnn6FrameEmb(sample_rate=32000, window_size=1024,
                           hop_size=320, mel_bins=64, fmin=50, fmax=14000,
                           classes_num=527)
checkpoint = torch.load(r"C:\Users\PDL Local User\Desktop\MTP\Cnn6_mAP=0.343.pth", map_location="cpu",
    weights_only=False)
frame_model.load_state_dict(checkpoint['model'])
frame_model.eval()




Cnn6FrameEmb(
  (spectrogram_extractor): Spectrogram(
    (stft): STFT(
      (conv_real): Conv1d(1, 513, kernel_size=(1024,), stride=(320,), bias=False)
      (conv_imag): Conv1d(1, 513, kernel_size=(1024,), stride=(320,), bias=False)
    )
  )
  (logmel_extractor): LogmelFilterBank()
  (spec_augmenter): SpecAugmentation(
    (time_dropper): DropStripes()
    (freq_dropper): DropStripes()
  )
  (bn0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv_block1): ConvBlock5x5(
    (conv1): Conv2d(1, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (conv_block2): ConvBlock5x5(
    (conv1): Conv2d(64, 128, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2), bias=False)
    (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (conv_block3): ConvBlock5x5(
    (conv1): Conv2d(128, 256, kernel_size=(5, 5)

In [6]:
path=r"C:\Users\90584\OneDrive\Desktop\Mtech_project\features\spectrogram"
top5=[("",0) for _ in range(5)]
for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith(".npy"):  # ensure only .npy files
            file_path = os.path.join(root, file)
            data = np.load(file_path)
            val=data.shape[2]
            print(f"{file},{val}")
            for i in range(5):
                if val>top5[i][1]:
                    top5[i+1:]=top5[i:-1]
                    top5[i]=(file,val)
                    break
print(top5)                                
                
            
            
    

file,303
file,303
file,303
file,152
file,114
file,152
file,152
file,303
file,227
file,303
file,145
file,181
file,146
file,133
file,139
file,173
file,130
file,213
file,174
file,103
file,67
file,117
file,222
file,158
file,161
file,172
file,6939
file,16169
file,411
file,493
file,1135
file,766
file,1846
file,588
file,1490
file,1531
file,1490
file,916
file,1641
file,1372
file,395
file,1029
file,403
file,403
file,5686
file,358
file,1029
file,455
file,723
file,403
file,679
file,3756
file,1088
file,470
file,301
file,281
file,236
file,252
file,211
file,257
file,230
file,414
file,154
file,257
file,186
file,282
file,282
file,253
file,261
file,263
file,244
file,311
file,193
file,226
file,199
file,258
file,286
file,251
file,245
file,225
file,209
file,296
file,233
file,317
file,246
file,216
file,256
file,258
file,205
file,208
file,203
file,225
file,192
file,199
file,194
file,216
file,199
file,287
file,177
file,214
file,212
file,230
file,190
file,225
file,243
file,338
file,247
file,208
file,219
file,

In [ ]:
spec=np.load(r"C:\files_mtp\MTP\features\spectrogram\atlantic_spotted_dolphin\6102500D_mel.npy")
print("Original shape:", spec.shape)

Original shape: (1, 128, 146)


In [67]:
# Convert to torch tensor
spec = torch.tensor(spec, dtype=torch.float32)

# Ensure shape is (batch, 1, time, mel)
spec = spec.unsqueeze(1)             # (batch, 1, mel, time)
spec = spec.permute(0, 1, 3, 2)      # (batch, 1, time, mel)

#print("Original shape:", spec.shape)

In [ ]:
# Example input spectrogram (batch=1, channel=1, time_steps=1024, mel_bins=64)
# below code during inference
with torch.no_grad():
    frame_embeddings = frame_model(spec)
frame_embeddings = frame_embeddings.mean(dim=3)  # average over freq

#print("Frame embedding shape:", frame_embeddings.shape)  # (batch, channels, time)

In [6]:
mfcc_feat=np.load(r"C:\files_mtp\MTP\features\mfcc\atlantic_spotted_dolphin\6102500D_mfcc.npy")
mfcc_feat=torch.tensor(mfcc_feat, dtype=torch.float32)

mfcc_feat.shape

torch.Size([1, 36, 146])

In [70]:
  # or unify CNN and MFCC dim
d_model=frame_embeddings.shape[1]
mfcc_shape=mfcc_feat.shape[1]
mfcc_feat = mfcc_feat.permute(0,2,1) 
# project MFCC to same dim
mfcc_proj = nn.Linear(mfcc_shape, d_model)
mfcc_feat_proj = mfcc_proj(mfcc_feat)   # [B, T, d_model]


frame_embeddings = frame_embeddings.permute(0,2,1)   
 
cross_attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=8, batch_first=True)
# cross attention: Q = cnn_feats, K,V = mfcc_feats
fused, _ = cross_attn(query=frame_embeddings, key=mfcc_feat_proj, value=mfcc_feat_proj)
#fused.shape  # [B, T, d_model]


In [73]:
fused.shape

torch.Size([1, 25, 512])

In [72]:

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        
        # Create matrix of shape (max_len, d_model)
        pe = torch.zeros(max_len, d_model)
        
        # Position indices [0, 1, 2, ...]
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # (max_len, 1)
        
        # Frequency terms for sine/cosine
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        # Apply sin to even indices; cos to odd indices
        pe[:, 0::2] = torch.sin(position * div_term)  # even dimensions
        pe[:, 1::2] = torch.cos(position * div_term)  # odd dimensions
        
        # Add batch dimension → shape (1, max_len, d_model)
        pe = pe.unsqueeze(0)  
        
        # Register as buffer so it's saved with model but not a parameter
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Tensor of shape (batch_size, seq_len, d_model)
        Returns:
            Tensor with positional encodings added
        """
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len, :]
        return x


In [74]:
Pos_fused=PositionalEncoding(d_model)(fused)

In [76]:
Pos_fused.shape

torch.Size([1, 25, 512])

In [77]:
# One encoder layer
encoder_layer = nn.TransformerEncoderLayer(d_model=512, nhead=8, batch_first=True)

# Full encoder stack (4 layers here)
transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=4)


In [78]:
encoded=transformer_encoder(Pos_fused)

In [79]:
encoded.shape

torch.Size([1, 25, 512])

In [80]:
x = encoded.mean(dim=1) 

In [81]:
x.shape

torch.Size([1, 512])

In [83]:
prob=nn.Linear(d_model, 55)(x)

In [84]:
prob

tensor([[-1.4421, -0.0627,  0.0054,  0.4628,  0.9216, -0.1381,  0.1068,  0.2125,
         -0.2969,  0.2515,  0.4451,  0.3134,  0.1642, -0.2236,  1.1211,  0.9550,
          0.6150,  0.0523,  0.5691,  0.8459, -0.7570,  0.0020,  0.3885,  0.4737,
          0.1530, -0.1705, -0.4144,  0.3846, -0.9849,  0.5228, -0.6707,  0.2379,
         -1.0886,  0.4887, -0.8188,  0.5808,  1.0743,  0.0704,  0.2708,  0.2252,
          0.0186, -0.4779,  0.6541, -0.1801, -0.7986,  1.0473,  0.0374,  0.0953,
          0.2698,  0.1084,  0.8215,  0.3103, -0.0875, -0.3834, -0.2554]],
       grad_fn=<AddmmBackward0>)